## ============================================================================

## E-COMMERCE FUNNEL ANALYTICS PROJECT
## NOTEBOOK: 01_DATA_PREPARATION

### PURPOSE:
### Clean the source event data and create analysis-ready fields for
### exploratory analysis and session-level funnel metrics.

## ============================================================================

## Preparation objectives

1. Load the source event dataset.
2. Create a working copy without changing the raw file.
3. Remove fully duplicated records.
4. Convert event timestamps to UTC datetime values.
5. Standardize missing category and brand values.
6. Create category hierarchy fields.
7. Remove records without a session identifier from the funnel dataset.
8. Export the cleaned event-level dataset.

In [10]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## Load source data

The raw file is kept unchanged. All preparation steps will be applied to a
separate working DataFrame.

In [11]:
events_raw = pd.read_csv("../data/events.csv")
events = events_raw.copy()

print(f"Source file: {"../data/events.csv"}")
print(f"Source rows: {len(events_raw):}")

Source file: ../data/events.csv
Source rows: 885129


## Remove fully duplicated records

The data-quality review identified fully duplicated rows. I will remove
exact duplicates while keeping the first occurrence of each record.

This step does not remove repeated users or repeated products. Those
repetitions represent normal e-commerce behavior.

In [12]:
rows_before_deduplication = len(events)

events = events.drop_duplicates().copy()

duplicates_removed = rows_before_deduplication - len(events)
print(f"Fully duplicated rows removed: {duplicates_removed:}")
print(f"Rows after deduplication: {len(events):}")

Fully duplicated rows removed: 655
Rows after deduplication: 884474


## Parse event timestamps

The source timestamp is stored as text. I will convert it to a UTC datetime
field so that the data can be sorted and grouped by date and hour.

In [13]:
events["event_time"] = pd.to_datetime(
    events["event_time"],
    utc=True,
    errors="coerce"
)

events["event_date"] = events["event_time"].dt.date
events["event_hour"] = events["event_time"].dt.hour

print(f"Invalid timestamps after conversion: {events['event_time'].isna().sum():,}")
events[["event_time", "event_date", "event_hour"]].head()

Invalid timestamps after conversion: 0


,event_time,event_date,event_hour
0,2020-09-24 11:57:06+00:00,2020-09-24,11
1,2020-09-24 11:57:26+00:00,2020-09-24,11
2,2020-09-24 11:57:27+00:00,2020-09-24,11
3,2020-09-24 11:57:33+00:00,2020-09-24,11
4,2020-09-24 11:57:36+00:00,2020-09-24,11


## Standardize missing category and brand values

Missing category and brand values will be retained as `Unknown`. Removing
these records would discard valid views, cart actions, or purchases from
the funnel analysis.

In [14]:
for column in ["category_code", "brand"]:
    events[column] = events[column].fillna("Unknown").astype("string").str.strip()
    events.loc[events[column].eq(""), column] = "Unknown"

print("Missing values:")
print(events[["category_code", "brand"]].isna().sum())

print("\nBlank values:")
print(events[["category_code", "brand"]].eq("").sum())

Missing values:
category_code    0
brand            0
dtype: int64

Blank values:
category_code    0
brand            0
dtype: Int64


## Create category hierarchy fields

The `category_code` field contains category levels separated by periods. I
will split it into separate fields to support category-level analysis.

In [15]:
category_levels = events["category_code"].str.split(".", n=2, expand=True)

events["category_level_1"] = category_levels[0].fillna("Unknown")
events["category_level_2"] = category_levels[1].fillna("Unknown")
events["category_level_3"] = category_levels[2].fillna("Unknown")

events[["category_code", "category_level_1", "category_level_2", "category_level_3"]].head()

,category_code,category_level_1,category_level_2,category_level_3
0,electronics.telephone,electronics,telephone,Unknown
1,computers.components.cooler,computers,components,cooler
2,Unknown,Unknown,Unknown,Unknown
3,computers.peripherals.printer,computers,peripherals,printer
4,Unknown,Unknown,Unknown,Unknown


## Handle missing session identifiers

The funnel will be calculated at session level. Records without a
`user_session` value cannot be assigned to a reliable session, so they will
be excluded from the analysis-ready funnel dataset.

In [16]:
missing_sessions = events["user_session"].isna().sum()
events = events.dropna(subset=["user_session"]).copy()

print(f"Rows without a session identifier removed: {missing_sessions:}")
print(f"Analysis-ready rows: {len(events):}")

Rows without a session identifier removed: 162
Analysis-ready rows: 884312


## Validate the analysis-ready dataset

I will confirm the final dimensions, event values, missing sessions, and
duplicate rows before exporting the cleaned file.

In [17]:
print(f"Final rows: {len(events):}")
print(f"Final columns: {len(events.columns)}")
print(f"Duplicate rows remaining: {events.duplicated().sum():}")
print(f"Missing sessions remaining: {events['user_session'].isna().sum():}")

print("\nEvent counts:")
print(events["event_type"].value_counts(dropna=False))

Final rows: 884312
Final columns: 14
Duplicate rows remaining: 0
Missing sessions remaining: 0

Event counts:
event_type
view        792943
cart         54026
purchase     37343
Name: count, dtype: int64


## Export cleaned data

The cleaned event-level dataset will be saved under `data/processed`. The
raw source file remains unchanged.

In [20]:
events.to_csv("../data/processed/events_clean.csv", index=False)

print(f"Cleaned dataset saved to: {"../data/processed/events_clean.csv"}")

Cleaned dataset saved to: ../data/processed/events_clean.csv
